# Lista 9

## Tensorboard and WandB

(5pkt + 2pkt)

Na liÅ›cie znajduje siÄ™ 1 zadanie. Po rozwiÄ…zaniu go, pokaÅ¼ kod prowadzÄ…cemu i odpowiedz na **pytanie kontrolne** â€” tylko wtedy przyznajemy punkty. Dodatkowo przeÅ›lij zadanie na platformie skos.

Dodatkowe zadanie oznaczone jest â­ï¸ i jest warte 2pkt.

## JeÅ›li Ä‡wiczenia bÄ™dÄ… siÄ™ przedÅ‚uaÅ‚y...

Odpowiedz na ponisze pytania pisemnie i przeÅ›lij zadanie na skos. Do zobaczenia na kolejnych zajÄ™ciach! ðŸ˜€

1. OtwÃ³rz Projector i uruchom jeden ze sposobÃ³w wizualizacji danych. Napisz ktÃ³ry uruchomiÅ‚eÅ›. Przeanalizuj embeddingi i naapisz co ciekawego zobaczyÅ‚eÅ›.
2. KtÃ³ra klasa na wykresie PR courve wypadÅ‚a najgorzej? Po czym to poznaÅ‚eÅ›/poznaÅ‚aÅ›?
3. OtwÃ³rz zakÅ‚adkÄ™, w ktÃ³rej zobaczysz graf sieci neuronowej. W jakie fragmenty sieci musisz "kliknÄ…Ä‡" w celu zobaczenia warstwy conv2?
4. (NieobowiÄ…zkowe dla obecnych, obowiÄ…zkowe dla nieobecnych) Dodaj tqdm do treningu sieci.

# TensorBoard 

## **1. Wczytanie danych i przygotowanie transformacji**

1. Wczytaj zbiÃ³r **CIFAR-10** (`torchvision.datasets.CIFAR10`).
2. Zastosuj transformacje:

   * `ToTensor()`
   * normalizacja kanaÅ‚Ã³w:
     `mean = (0.5, 0.5, 0.5)`
     `std  = (0.5, 0.5, 0.5)`
3. Przygotuj **DataLoader** dla train i test z `batch_size=4`.

---

## **2. Uruchomienie TensorBoard + zapis przykÅ‚adowych obrazÃ³w**

1. Zainicjalizuj logger:
   `writer = SummaryWriter('runs/cifar10_experiment')`
2. Pobierz jeden batch z trainloadera i:

   * stwÃ³rz grid (`make_grid`)
   * zapisz go przez `writer.add_image(...)`.

---

## **3. Wizualizacja architektury modelu**

1. Zaimplementuj prostÄ… CNN (2Ã—Conv2d, 3Ã—Linear).
2. UÅ¼yj przykÅ‚adowego batcha z DataLoadera i zapisz graf:
   `writer.add_graph(model, images)`.

---

## **4. Embedding â€“ 100 losowych obrazÃ³w**

1. Wybierz 100 losowych przykÅ‚adÃ³w z train set (numpy â†’ tensor).
2. ZamieÅ„ format (N, H, W, C) â†’ (N, C, H, W).
3. SpÅ‚aszcz kaÅ¼dy obraz do wymiaru **3072**.
4. Zapisz embedding (`add_embedding`):

   * `metadata = nazwy klas`,
   * `label_img = obrazki`.

---

## **5. Åšledzenie treningu**

1. PrzeprowadÅº jednÄ… epokÄ™ treningu.
2. Co 500 iteracji:

   * `writer.add_scalar('training_loss', loss)`
   * stwÃ³rz figure Matplotlib z predykcjami vs. prawdziwe etykiety i zapisz przez `add_figure`.

---

## **6. Ocena modelu â€“ Precision-Recall**

1. Przelicz softmax na caÅ‚ym zbiorze testowym.
2. Dla kaÅ¼dej klasy (0â€“9):

   * przygotuj maskÄ™ prawdy i wektor prawdopodobieÅ„stw,
   * zapisz wykres PR: `writer.add_pr_curve(...)`.

---

## **7. ZakoÅ„czenie**

Na koÅ„cu wywoÅ‚aj `writer.close()`.
TensorBoard uruchom komendÄ…:

```bash
tensorboard --logdir=runs
```

In [8]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import matplotlib.pyplot as plt
import numpy as np
from torch.utils.tensorboard import SummaryWriter

# ================================================================
# 1. WCZYTANIE DANYCH + TRANSFORMACJE
# ================================================================

# Transformacje: konwersja do tensora + normalizacja kanaÅ‚Ã³w RGB
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# Wczytanie zbioru CIFAR-10 (train i test)
trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)
testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# DataLoadery â€“ batch_size = 4 (dla przejrzystoÅ›ci w TensorBoard)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=4, shuffle=True
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=4, shuffle=False
)

classes = trainset.classes  # nazwy klas CIFAR-10

# ================================================================
# Helper: wyÅ›wietlanie obrazÃ³w w figure Matplotlib
# ================================================================
def matplotlib_imshow(img):
    # "odwrÃ³cenie" normalizacji: [-1,1] -> [0,1]
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1,2,0)))

# ================================================================
# 2. TWORZENIE I URUCHAMIANIE TENSORBOARD + ZAPIS PIERWSZYCH OBRAZÃ“W
# ================================================================

# Inicjalizacja loggera TensorBoard
writer = SummaryWriter('runs/cifar10_experiment')

# Pobranie jednego batcha i zapis jako obraz (grid)
dataiter = iter(trainloader)
images, labels = next(dataiter)
img_grid = torchvision.utils.make_grid(images)

# Zapis obrazÃ³w do TensorBoard (TensorBoard â†’ Images)
# UzupeÅ‚nij (add_imade)


# write to tensorboard
writer.add_image('cifar10_images', img_grid)

# ================================================================
# 3. DEFINICJA I WIZUALIZACJA ARCHITEKTURY SIECI
# ================================================================

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Prosta CNN: 2 warstwy konwolucyjne + 3 w peÅ‚ni poÅ‚Ä…czone
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 1. Conv + ReLU + Pool
        x = self.pool(F.relu(self.conv2(x)))  # 2. Conv + ReLU + Pool
        x = x.view(-1, 16*5*5)                # spÅ‚aszczenie
        x = F.relu(self.fc1(x))               # FC1
        x = F.relu(self.fc2(x))               # FC2
        x = self.fc3(x)                       # FC3 (wyjÅ›cie logits)
        return x

net = Net()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

# Zapis grafu modelu do TensorBoard
# (TensorBoard â†’ Graph)
# UzupeÅ‚nij (add_graph)
writer.add_graph(net, images)

# ================================================================
# 4. EMBEDDING 100 LOSOWYCH OBRAZKÃ“W
# ================================================================

# Helper: wybÃ³r losowych obrazkÃ³w i ich etykiet
def select_n_random(data, labels, n=100):
    assert len(data) == len(labels)
    perm = torch.randperm(len(data))
    return data[perm][:n], torch.tensor(labels)[perm][:n]

# WybÃ³r 100 elementÃ³w
images, labels = select_n_random(trainset.data, trainset.targets)

# Zamiana z numpy (N,H,W,C) â†’ tensor (N,C,H,W)
images = torch.tensor(images).permute(0, 3, 1, 2).float()

# Zamiana etykiet na nazwy klas
class_labels = [classes[l] for l in labels]

# SpÅ‚aszczenie obrazÃ³w do wymiaru 3072 (32Ã—32Ã—3)
features = images.view(100, -1)

# Zapis do TensorBoard (â†’ zakÅ‚adka Projector)
# UzupeÅ‚nij (add_embedding, label_img=images/255.0)
writer.add_embedding(
    features,
    metadata=class_labels,
    label_img=images/255.0,
    global_step=1,
    tag='cifar10_embedding'
)

# ================================================================
# 5. ÅšLEDZENIE TRENINGU (SCALARS + IMAGES)
# ================================================================

# Helper: obliczanie predykcji + prawdopodobieÅ„stw
def images_to_probs(net, images):
    output = net(images)
    _, preds = torch.max(output, 1)  # predykcje
    # wyciÄ…gniÄ™cie prawdopodobieÅ„stwa kaÅ¼dej przewidzianej klasy
    probs = [F.softmax(o, dim=0)[p].item() for o,p in zip(output, preds)]
    return preds, probs

# Helper: tworzenie figure z predykcjami vs. etykiety
def plot_classes_preds(net, images, labels):
    preds, probs = images_to_probs(net, images)
    fig = plt.figure(figsize=(12,4))

    for i in range(4):
        ax = fig.add_subplot(1,4,i+1)
        matplotlib_imshow(images[i])
        ax.set_title(
            f"{classes[preds[i]]}: {probs[i]*100:.1f}%\n"
            f"(label: {classes[labels[i]]})",
            color=("green" if preds[i] == labels[i] else "red")
        )
        ax.axis('off')
    return fig

# ------------------------------
# PÄ™tla treningowa: 1 epoka
# co 500 batchy zapis do TensorBoard
# ------------------------------
running_loss = 0.0
for epoch in range(1):
    for i, (inputs, labels) in enumerate(trainloader):

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # zapis co 500 iteracji
        if i % 500 == 499:
            # Zapis training loss
            # UzupeÅ‚nij (add_scalar)
            avg_loss = running_loss / 500
            writer.add_scalar(
                'training loss',
                avg_loss,
                epoch * len(trainloader) + i
            )

            # Images â†’ predykcje vs rzeczywiste etykiety
            fig = plot_classes_preds(net, inputs, labels)
            # UzupeÅ‚nij (add_figure)
            writer.add_figure('predictions_vs_labels', fig, global_step=epoch * len(trainloader) + i)
            running_loss = 0.0

# ================================================================
# 6. KRZYWE PRECISIONâ€“RECALL DLA 10 KLAS
# ================================================================

class_probs = []
class_label = []

# Zebranie predykcji na danych testowych
with torch.no_grad():
    for images, labels in testloader:
        outputs = net(images)
        probs = F.softmax(outputs, dim=1)
        class_probs.append(probs)
        class_label.append(labels)

test_probs = torch.cat(class_probs)   # macierz [N,10]
test_label = torch.cat(class_label)   # wektor [N]

# Funkcja zapisujÄ…ca PR curve jednej klasy
def add_pr_curve(class_index, global_step=0):
    truth = (test_label == class_index)     # maska prawdziwych przykÅ‚adÃ³w
    probs = test_probs[:, class_index]      # prawdopodobieÅ„stwa tej klasy

    # PR curve do TensorBoard
    # UzupeÅ‚nij (add_pr_curve(classes[class_index]...))
    writer.add_pr_curve(
        classes[class_index],
        truth,
        probs,
        global_step=global_step
    )


# PR curves dla wszystkich klas CIFAR-10
for i in range(10):
    add_pr_curve(i, global_step=0)

# Zamykamy logger

writer.close()

# â­ï¸ Weights & Biases

Twoim zadaniem jest stworzenie **mini-eksperymentu machine learningowego**, ktÃ³ry bÄ™dzie w peÅ‚ni rejestrowany w systemie **Weights & Biases (W&B)**.

---

### 1ï¸ Zainstaluj i poÅ‚Ä…cz siÄ™ z W&B

UÅ¼yj `wandb.login()`, aby poÅ‚Ä…czyÄ‡ siÄ™ ze swoim kontem.

---

### 2ï¸ Przygotuj sÅ‚ownik hiperparametrÃ³w

UmieÅ›Ä‡ tam m.in.:

* liczbÄ™ epok,
* batch size,
* learning rate,
* informacjÄ™ o architekturze i zbiorze danych.

To wszystko zostanie automatycznie zarejestrowane w W&B.

---

### 3ï¸ Zaimplementuj pipeline treningowy

Napisz funkcjÄ™ `model_pipeline(config)`, ktÃ³ra:

* **inicjalizuje eksperyment** w W&B,
* tworzy dane, model, loss i optimizer,
* trenuje model i **loguje metryki**,
* testuje go na zbiorze testowym,
* **zapisuje model** do formatu ONNX i wrzuca go do W&B.

---

### 4ï¸ Loguj rÃ³Å¼ne rzeczy do W&B

Przynajmniej:

* stratÄ™ podczas treningu (`loss`),
* numer epoki,
* parametry i gradienty modelu (poprzez `run.watch()`).
* dokÅ‚adnoÅ›Ä‡ na danych testowych (po treningu)

---

### 5 Uruchom eksperyment i odwiedÅº stronÄ™ W&B

SprawdÅº:

* wykres strat,
* gradienty i parametry,
* zapisany model,
* podsumowanie runu.

In [ ]:
# ================================
# 1. Importy i przygotowanie Å›rodowiska
# ================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm
import wandb

# Ustalanie staÅ‚ych seedÃ³w, aby wyniki byÅ‚y powtarzalne
torch.backends.cudnn.deterministic = True
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

# Automatyczny wybÃ³r GPU, jeÅ›li dostÄ™pne
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Logowanie do platformy Weights & Biases
# UzupeÅ‚nij


# ================================
# 2. Konfiguracja hiperparametrÃ³w
# ================================

# UzupeÅ‚nij (config:
# epochs=3,                # liczba epok
# batch_size=128,          # rozmiar batcha
# learning_rate=0.001,     # learning rate
# dataset="MNIST",         # uÅ¼ywany zbiÃ³r danych
# architecture="SimpleCNN",# meta-info o modelu
# classes=10,              # liczba klas w MNIST
# kernels=[16, 32]         # liczba filtrÃ³w w warstwach CNN)



# ================================
# 3. Funkcje pomocnicze â€” dane
# ================================

def get_data(train=True):
    """
    Pobiera zbiÃ³r MNIST i konwertuje go na tensory.
    """
    dataset = torchvision.datasets.MNIST(
        root=".",
        train=train,
        transform=transforms.ToTensor(),
        download=True
    )
    return dataset


def make_loader(dataset, batch_size):
    """
    Tworzy DataLoader dla MNIST.
    """
    return torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )


# ================================
# 4. Definicja modelu CNN
# ================================

class ConvNet(nn.Module):
    """
    Prosta, dwuwarstwowa sieÄ‡ konwolucyjna + warstwa w peÅ‚ni poÅ‚Ä…czona.
    """
    def __init__(self, kernels, classes=10):
        super().__init__()

        # Pierwsza warstwa konwolucyjna
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, kernels[0], kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Druga warstwa konwolucyjna
        self.layer2 = nn.Sequential(
            nn.Conv2d(kernels[0], kernels[1], kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Warstwa w peÅ‚ni poÅ‚Ä…czona
        self.fc = nn.Linear(7 * 7 * kernels[1], classes)

    def forward(self, x):
        """
        Definicja przepÅ‚ywu danych w sieci.
        """
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)


# ================================
# 5. Funkcja treningowa â€” logowanie W&B
# ================================

def train(model, loader, criterion, optimizer, config, run):
    """
    Trenuje model i loguje metryki do W&B.
    """

    # Automatyczne logowanie gradientÃ³w i wag modelu
    # UzupeÅ‚nij (watch)

    for epoch in range(config.epochs):
        for batch, (images, labels) in enumerate(loader):

            # Przeniesienie danych na GPU (jeÅ›li dostÄ™pne)
            images, labels = images.to(device), labels.to(device)

            # Zerowanie gradientÃ³w
            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()

            # Aktualizacja wag
            optimizer.step()

            # Logowanie strat co 50 batchy
            if batch % 50 == 0:
                # UzupeÅ‚nij (log: loss, epoch)


# ================================
# 6. Funkcja testowa + zapis modelu ONNX
# ================================

def test(model, loader, run):
    """
    Testuje model oraz zapisuje jego parametry na W&B.
    """

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            _, pred = torch.max(outputs, 1)

            # Statystyki dokÅ‚adnoÅ›ci
            total += labels.size(0)
            correct += (pred == labels).sum().item()

    accuracy = correct / total

    # Logowanie dokÅ‚adnoÅ›ci testowej
    # UzupeÅ‚nij (log: accuracy)
    print("Test accuracy:", accuracy)

    # Eksport modelu do ONNX
    dummy_input = torch.randn(1, 1, 28, 28).to(device)
    # UzupeÅ‚nij (exportowanie modelu do formatu ONNX)

    # Zapisanie modelu w W&B
    # UzupeÅ‚nij (zapis modelu do W&B)


# ================================
# 7. PeÅ‚ny pipeline eksperymentu
# ================================

def model_pipeline(hyperparameters):
    """
    GÅ‚Ã³wna funkcja kontrolujÄ…ca caÅ‚y proces:
    - inicjalizacja W&B
    - trening
    - testowanie
    - zapis modelu
    """

    # Inicjalizacja eksperymentu
    with # UzupeÅ‚nij (wband init... ) as run:

        config = run.config  # Wersja configu synchronizowana z W&B

        # 1. Przygotowanie danych
        train_dataset = get_data(train=True)
        test_dataset = get_data(train=False)
        train_loader = make_loader(train_dataset, config.batch_size)
        test_loader = make_loader(test_dataset, config.batch_size)

        # 2. Utworzenie modelu
        model = ConvNet(config.kernels, config.classes).to(device)

        # 3. Loss + optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

        # 4. Trening
        train(model, train_loader, criterion, optimizer, config, run)

        # 5. Testowanie + zapis modelu
        test(model, test_loader, run)

    return model


# ================================
# 8. Uruchomienie eksperymentu
# ================================

model = model_pipeline(config)